# Last 10,000 news articles — persistent Drive checkpoints
Run with a Colab GPU. Mount Drive when prompted. Original attributes and full text are preserved. Progress CSV is updated every 100 articles.


In [ ]:
# Generated with assistance from ChatGPT. Labels are model-generated, not verified ground truth.
from pathlib import Path
from google.colab import drive
import torch
assert torch.cuda.is_available(), 'GPU unavailable. No labelling started. Connect a GPU runtime, then rerun this cell.'
drive.mount('/content/drive')
assert Path('/content/drive/MyDrive').exists(), 'Drive must be mounted; no temporary-storage fallback.'

from pathlib import Path
import pandas as pd
import csv, io, json, zipfile, urllib.request

# Exact upstream snapshot used to prepare our dataset.
COMMIT = '8a14da7c922131fcf6adc4f70726cabcb1c518d8'
archive_path = Path('/content/lk_news_snapshot.zip')
INPUT_PATH = Path('/content/news_source/news_english.csv')
INPUT_PATH.parent.mkdir(exist_ok=True)
if not INPUT_PATH.exists():
    if not archive_path.exists():
        print('Downloading the pinned source archive...', flush=True)
        urllib.request.urlretrieve(f'https://codeload.github.com/nuuuwan/lk_news/zip/{COMMIT}', archive_path)
    print('Constructing the English CSV from metadata and article text...', flush=True)
    with zipfile.ZipFile(archive_path) as z:
        names = z.namelist()
        manifest_path = next(n for n in names if n.endswith('/data/lk_news/docs_all.tsv'))
        metadata = list(csv.DictReader(io.StringIO(z.read(manifest_path).decode('utf-8-sig')), delimiter='	'))
        paths = {n.split('/')[-2]: n.rsplit('/',1)[0] for n in names if n.endswith('/doc.json')}
        records = []
        for m in metadata:
            if m['lang'] == 'en':
                p = paths[m['doc_id']]
                r = json.loads(z.read(p + '/doc.json'))
                r['text'] = z.read(p + '/doc.txt').decode('utf-8')
                records.append(r)
    pd.DataFrame(records).to_csv(INPUT_PATH, index=False, encoding='utf-8')
    del records, metadata, paths, names

df = pd.read_csv(INPUT_PATH, keep_default_na=False)
assert len(df) == 51590
print(f'Loaded {len(df):,} articles and {len(df.columns)} columns.')
display(df.drop(columns=['text']).head())

# Generated with assistance from ChatGPT; reviewed/adapted for this assignment.
import hashlib
import re
import unicodedata

MIN_WORDS = 50
MIN_LATIN_RATIO = 0.80
original_columns = df.columns.tolist()
required = {'doc_id', 'date_str', 'lang', 'text'}
assert required.issubset(df.columns), f"Missing columns: {required - set(df.columns)}"
assert df['doc_id'].is_unique, "Review duplicate document IDs before continuing."

def normalized_text_hash(text):
    normalized = unicodedata.normalize('NFKC', str(text))
    normalized = re.sub(r'\s+', ' ', normalized).strip().casefold()
    return hashlib.sha256(normalized.encode('utf-8')).hexdigest()

def latin_letter_ratio(text):
    letters = [char for char in str(text) if char.isalpha()]
    if not letters:
        return 0.0
    latin_letters = sum('LATIN' in unicodedata.name(char, '') for char in letters)
    return latin_letters / len(letters)

# Sort deterministically: retain the earliest record in each exact duplicate group.
working = df.sort_values(['date_str', 'doc_id'], kind='stable').copy()
word_counts = working['text'].str.split().str.len()
latin_ratios = working['text'].map(latin_letter_ratio)
text_hashes = working['text'].map(normalized_text_hash)

# These are reproducible quality-screening rules, not perfect language detection.
flags = pd.DataFrame({
    'not_tagged_english': working['lang'].ne('en'),
    'short_text': word_counts.lt(MIN_WORDS),
    'language_review': latin_ratios.lt(MIN_LATIN_RATIO),
    'replacement_character': working['text'].str.contains('\ufffd', regex=False),
    'normalized_exact_duplicate': text_hashes.duplicated(keep='first'),
}, index=working.index)

summary = flags.sum().rename('flagged_articles').to_frame()
print("Flag counts can overlap:")
display(summary)

remove_mask = flags.any(axis=1)
cleaned_df = working.loc[~remove_mask, original_columns].reset_index(drop=True)
print(f"Input:   {len(df):,}")
print(f"Removed: {int(remove_mask.sum()):,} unique articles")
print(f"Kept:    {len(cleaned_df):,}")

# Preserve source text and every original attribute; only filter rows.
assert cleaned_df.columns.tolist() == original_columns
assert cleaned_df['doc_id'].is_unique
assert cleaned_df['lang'].eq('en').all()
assert not cleaned_df['text'].map(normalized_text_hash).duplicated().any()
assert len(cleaned_df) + int(remove_mask.sum()) == len(df)

OUTPUT_PATH = Path('news_english_cleaned.csv')
cleaned_df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')
saved = pd.read_csv(OUTPUT_PATH, keep_default_na=False)
pd.testing.assert_frame_equal(saved, cleaned_df)
print(f"Saved and verified: {OUTPUT_PATH.resolve()}")


# Last 10,000 rows in cleaned CSV order (sorted by date_str, then doc_id).
# This is a recent chronological subset, not a representative random sample.
import numpy as np
import os, time
from transformers import pipeline
from tqdm.auto import tqdm
df = cleaned_df.tail(10000).reset_index(drop=True).copy()
assert len(df) == 10000 and df['doc_id'].is_unique
RUN_DIR = Path('/content/drive/MyDrive/LankaNews/zero_shot_v8_last10000')
RUN_DIR.mkdir(parents=True, exist_ok=True)
INPUT_PATH = RUN_DIR / 'news_english_last10000_input.csv'
if INPUT_PATH.exists():
    pd.testing.assert_frame_equal(pd.read_csv(INPUT_PATH, keep_default_na=False), df)
else:
    df.to_csv(INPUT_PATH, index=False)
print('Selected last 10,000:', df['date_str'].min(), 'to', df['date_str'].max(), flush=True)

MODEL_ID = 'facebook/bart-large-mnli'
TOPICS = {
    'politics_governance': 'politics and government',
    'business_economy': 'business and economics',
    'crime_justice': 'crime and court cases',
    'sport': 'sports',
    'health_healthcare': 'Health & Healthcare',
    'weather_environment_disasters': 'Weather, Environment & Disasters',
    'tourism_travel': 'Tourism & Travel',
    'transport': 'transport and vehicles',
    'telecommunications': 'telecommunications and internet connectivity',
    'information_technology': 'software, computing or artificial intelligence',
    'education': 'education',
    'entertainment': 'entertainment and show business',
    'other': 'lifestyle, culture, religion or general-interest news',
}
# Score simple alternative subjects, then take the strongest match per topic.
ALIASES = {
    'politics': 'politics_governance',
    'government and diplomacy': 'politics_governance',
    'business': 'business_economy',
    'economics': 'business_economy',
    'crime': 'crime_justice',
    'legal proceedings': 'crime_justice',
    'sports': 'sport',
    'cricket': 'sport',
    'football': 'sport',
    'medicine and healthcare': 'health_healthcare',
    'weather': 'weather_environment_disasters',
    'environmental protection and pollution': 'weather_environment_disasters',
    'natural disasters': 'weather_environment_disasters',
    'transport': 'transport',
    'vehicles': 'transport',
    'telecommunications': 'telecommunications',
    'information technology': 'information_technology',
    'education': 'education',
    'entertainment': 'entertainment',
    'music': 'entertainment',
    'films and television': 'entertainment',
    'tourism and travel': 'tourism_travel',
    'hotels and holiday destinations': 'tourism_travel',
    'religion and spirituality': 'other',
    'lifestyle and human interest': 'other',
}
TEMPLATE = 'This text is about {}.'
MODEL_REVISION = 'd7645e127eaf1aefc7862fd59a17a5aa8558b8ce'
assert torch.cuda.is_available(), 'Select a Colab GPU runtime before the full run.'
classifier = pipeline('zero-shot-classification', model=MODEL_ID,
                      revision=MODEL_REVISION, device=0, torch_dtype=torch.float16)
REVISION = getattr(classifier.model.config, '_commit_hash', None) or 'unknown'
BATCH_SIZE = 16
print('Device:', classifier.device, '| model revision:', REVISION)

# Heuristics for review, NOT validated accuracy thresholds.
MIN_SCORE = 0.70
MIN_MARGIN = 0.20

def predict_topics(frame):
    inputs = [str(title) + '\n' + ' '.join(str(text).split()[:200])
              for title, text in zip(frame['description'], frame['text'])]
    global BATCH_SIZE
    while True:
        try:
            results = classifier(inputs, candidate_labels=list(ALIASES),
                                 hypothesis_template=TEMPLATE, multi_label=True,
                                 batch_size=BATCH_SIZE)
            break
        except torch.cuda.OutOfMemoryError:
            if BATCH_SIZE == 1:
                raise
            BATCH_SIZE = max(1, BATCH_SIZE // 2)
            torch.cuda.empty_cache()
            print('Reduced GPU batch size to', BATCH_SIZE, flush=True)
    rows = []
    for doc_id, result in zip(frame['doc_id'], results):
        alias_scores = dict(zip(result['labels'], result['scores']))
        scores = {topic: max(alias_scores[a] for a, group in ALIASES.items() if group == topic)
                  for topic in TOPICS}
        ranked = sorted(scores, key=scores.get, reverse=True)
        score, second_score = scores[ranked[0]], scores[ranked[1]]
        row = {
            'doc_id': doc_id,
            'predicted_topic': ranked[0],
            'matched_phrase': max((a for a, group in ALIASES.items() if group == ranked[0]), key=alias_scores.get),
            'teacher_score': score,
            'teacher_margin': score - second_score,
            'needs_review': score < MIN_SCORE or score - second_score < MIN_MARGIN,
            'label_source': 'zero_shot_pseudo_label',
            'teacher_model': MODEL_ID,
            'teacher_revision': REVISION,
            'taxonomy_version': 'v8',
            'input_policy': 'headline_plus_first_200_body_words',
            'scoring_policy': 'independent_alias_scores_max_per_topic',
        }
        row.update({'score_' + topic: scores[topic] for topic in TOPICS})
        rows.append(row)
    return pd.DataFrame(rows)

assert len(TOPICS) == 13
assert set(ALIASES.values()) == set(TOPICS)

# Saves each completed 100-row chunk atomically; safe to rerun with the same input/config.
# Existing mounted Drive is preferred. Otherwise /content is temporary runtime storage.
RUN_DIR = Path('/content/drive/MyDrive/LankaNews/zero_shot_v8_last10000')
RUN_DIR.mkdir(parents=True, exist_ok=True)
PARTS = RUN_DIR / 'checkpoints'
PARTS.mkdir(exist_ok=True)
OUTPUT = RUN_DIR / 'news_english_pseudo_labelled_v8_last10000.csv'

input_hash = hashlib.sha256()
with INPUT_PATH.open('rb') as handle:
    for block in iter(lambda: handle.read(1024 * 1024), b''):
        input_hash.update(block)
settings = {'model': MODEL_ID, 'revision': REVISION, 'topics': TOPICS, 'aliases': ALIASES,
            'template': TEMPLATE, 'taxonomy': 'v8', 'input_sha256': input_hash.hexdigest(),
            'input_policy': 'headline_plus_first_200_body_words', 'precision': 'float16',
            'min_score': MIN_SCORE, 'min_margin': MIN_MARGIN,
            'scoring_policy': 'independent_alias_scores_max_per_topic', 'chunk_size': 100}
config_id = hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()
manifest_path = RUN_DIR / 'run_settings.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == settings, 'Input/config changed: use a new run directory.'
else:
    manifest_path.write_text(json.dumps(settings, indent=2))

# Small execution check before launching all articles; not an accuracy evaluation.
smoke = predict_topics(df.iloc[:3])
score_columns = ['score_' + topic for topic in TOPICS]
assert np.isfinite(smoke[score_columns].to_numpy()).all()
assert smoke['predicted_topic'].isin(TOPICS).all()
print('GPU execution check passed. Categories:', len(TOPICS), '| device:', classifier.device, flush=True)
print('Checkpoint/output folder:', RUN_DIR, flush=True)
print('Storage:', 'Google Drive' if '/drive/MyDrive/' in str(RUN_DIR) else 'temporary Colab runtime; download results before resetting', flush=True)

started = time.monotonic()
for start in tqdm(range(0, len(df), 100), desc='Last 10,000 (100 articles per chunk)'):
    frame = df.iloc[start:start + 100]
    part = PARTS / f'part_{start:06d}.csv'
    if part.exists():
        saved = pd.read_csv(part, keep_default_na=False)
        assert saved['doc_id'].tolist() == frame['doc_id'].tolist()
        assert saved['run_config_id'].eq(config_id).all()
        assert np.isfinite(saved[score_columns].to_numpy()).all()
        continue
    result = predict_topics(frame)
    assert len(result) == len(frame)
    assert result['doc_id'].tolist() == frame['doc_id'].tolist()
    assert np.isfinite(result[score_columns].to_numpy()).all()
    result['run_config_id'] = config_id
    temp = part.with_suffix('.csv.tmp')
    result.to_csv(temp, index=False)
    os.replace(temp, part)
    partial_predictions = pd.concat([pd.read_csv(p, keep_default_na=False) for p in sorted(PARTS.glob('part_*.csv'))], ignore_index=True)
    partial = df.merge(partial_predictions, on='doc_id', how='inner', validate='one_to_one')
    partial_path = RUN_DIR / 'news_english_last10000_progress.csv'
    partial.to_csv(partial_path.with_suffix('.csv.tmp'), index=False)
    os.replace(partial_path.with_suffix('.csv.tmp'), partial_path)
    completed = min(start + 100, len(df))
    if start == 0 or completed % 1000 == 0 or completed == len(df):
        print(f'Checkpointed {completed:,}/{len(df):,} articles; elapsed {(time.monotonic()-started)/60:.1f} min', flush=True)

predictions = pd.concat([pd.read_csv(p, keep_default_na=False) for p in sorted(PARTS.glob('part_*.csv'))], ignore_index=True)
assert len(predictions) == len(df) and predictions['doc_id'].is_unique
assert set(predictions['doc_id']) == set(df['doc_id'])
assert predictions['predicted_topic'].isin(TOPICS).all()
labelled_df = df.merge(predictions, on='doc_id', how='left', validate='one_to_one')
assert labelled_df[df.columns].equals(df)
labelled_df.to_csv(OUTPUT, index=False)
labelled_df.to_csv(OUTPUT.with_suffix('.csv.gz'), index=False, compression='gzip')
print('COMPLETE:', len(labelled_df), 'articles saved to', OUTPUT, flush=True)
display(labelled_df['predicted_topic'].value_counts().rename('articles').to_frame())
print('Review flags:', labelled_df['needs_review'].sum())
